In [0]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from pyspark.sql import SparkSession
from pyspark.sql.functions import max

def get_latest_metrics(spark, base_adls):
    # Read the Gold financial summary
    gold_df = spark.read.format("delta").load(f"{base_adls}/delta/gold_financial_summary_daily")
    
    # Get the most recent service date
    latest_date_row = gold_df.select(max("service_date").alias("latest")).collect()[0]
    latest_date = latest_date_row["latest"]
    
    # Filter for only the latest day's records and collect them to the driver
    daily_records = gold_df.filter(gold_df.service_date == latest_date).collect()
    
    return latest_date, daily_records

def send_email_alert(latest_date, daily_records):
    # Securely retrieve Gmail credentials using Databricks Secrets
    # Note: dbutils is automatically available in Databricks environments
    sender_email = dbutils.secrets.get(scope="smtp_creds", key="email_address")
    app_password = dbutils.secrets.get(scope="smtp_creds", key="app_password")
    receiver_email = "stakeholders@yourcompany.com"

    # Format the email body
    html_content = f"""
    <html>
      <body>
        <h2>Daily Claims Financial Summary</h2>
        <p><strong>Date:</strong> {latest_date}</p>
        <table border="1" cellpadding="5">
          <tr>
            <th>Source System</th>
            <th>Total Claims</th>
            <th>Total Revenue ($)</th>
          </tr>
    """
    
    for row in daily_records:
        html_content += f"""
          <tr>
            <td>{row['source_system']}</td>
            <td>{row['total_claim_count']}</td>
            <td>{row['total_revenue']}</td>
          </tr>
        """
        
    html_content += """
        </table>
      </body>
    </html>
    """

    # Construct the email message
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = f"Automated Alert: Gold Layer Financial Summary ({latest_date})"
    msg.attach(MIMEText(html_content, 'html'))

    # Connect to Gmail SMTP server and send
    print("Connecting to SMTP server...")
    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(sender_email, app_password)
        server.send_message(msg)
        print("Alert email sent successfully.")

if __name__ == "__main__":
    spark = SparkSession.builder.getOrCreate()
    base_adls = "abfss://landing-zone@lshc.dfs.core.windows.net"
    
    latest_date, daily_records = get_latest_metrics(spark, base_adls)
    send_email_alert(latest_date, daily_records)